# 01-Data Generation



In [1]:
from faker import Faker
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

fake = Faker('es_ES')


In [7]:
hoy = datetime.now()
hace_un_anio = hoy - timedelta(days=365)
hace_un_anio

datetime.datetime(2025, 5, 13, 12, 22, 20, 21132)

## Generate customers
- customer_id
- company_name
- industry
- country
- plan_type
- mrr
- contract_start
- contract_end
- churned
- churn_date
- employee_count
- health_score

In [93]:
def generar_cliente():
    rangos_mmr = {
        "Starter": (99, 299),
        "Pro": (300, 999),
        "Enterprise": (1000, 5000)
    }
    plan_type = random.choice(["Starter", "Pro", "Enterprise"])
    churned = fake.boolean()
    contract_start = fake.date_between(start_date="-4y", end_date='today')
    contract_end = contract_start + timedelta(days=30*random.choice([1,3,6,12]))
    if churned == True: 
        churned_date = fake.date_between(start_date=contract_start, end_date=contract_end)
        health_score = random.randint(0,50)
    else:
        churned_date = None
        health_score = random.randint(40,100)


    return {
        "customer_id":      fake.uuid4(),
        "company_name":     fake.company(),
        "industry":         random.choice(["Tech", "Retail", "Finance", "Health"]),
        "country":          fake.country(),
        "plan_type":        plan_type,
        "mrr":              random.randint(*rangos_mmr[plan_type]),
        "contract_start":   contract_start,
        "contract_end":     contract_end,
        "churned":          churned,
        "churned_date":     churned_date,
        "employee_count":   random.randint(10,5000),
        "health_score":     health_score
    }

print(generar_cliente())

{'customer_id': 'c58b51b5-12cf-4a03-a874-1ca056c325ce', 'company_name': 'Armengol & Asociados S.L.L.', 'industry': 'Retail', 'country': 'El Salvador', 'plan_type': 'Pro', 'mrr': 654, 'contract_start': datetime.date(2025, 10, 19), 'contract_end': datetime.date(2026, 4, 17), 'churned': True, 'churned_date': datetime.date(2026, 2, 5), 'employee_count': 666, 'health_score': 32}


In [94]:
customers = [generar_cliente() for _ in range(3000)]

In [96]:
df_customers = pd.DataFrame(data=customers)
df_customers

,customer_id,company_name,industry,country,plan_type,mrr,contract_start,contract_end,churned,churned_date,employee_count,health_score
0,949c8a0e-c8f2-40cc-8f97-527bcd35406a,Construcción Castellana S.Coop.,Health,Iraq,Enterprise,4972,2023-12-19,2024-03-18,True,2024-01-16,108,10
1,bb69a3c1-816c-454b-8d94-481186e8c99a,Castells y asociados S.Com.,Retail,Hungría,Starter,135,2025-07-26,2026-07-21,False,None,1532,91
2,5220a66c-36b8-4037-8c43-b08eb81203e2,Riquelme & Asociados S.Com.,Finance,Túnez,Enterprise,4468,2024-02-23,2024-03-24,False,None,909,97
3,b6878b37-e5da-460d-9d8b-aafa4eb5b029,Alberto y asociados S.Com.,Health,Argelia,Pro,846,2022-08-13,2023-08-08,False,None,3559,93
4,3439ad77-7bbb-41c9-88f8-1f276cf29283,Úrsula Lopez Morante S.Coop.,Retail,Niger,Pro,848,2023-01-31,2023-07-30,False,None,4096,86
...,...,...,...,...,...,...,...,...,...,...,...,...
2995,24d727ee-2ce0-4db9-a64f-e82458f8a86f,Aranda y Becerra S.L.,Health,Venezuela,Enterprise,1832,2023-01-17,2023-07-16,True,2023-02-22,1230,9
2996,dae3f348-c69f-4eae-bbe0-c5e393987c9b,Marcial Poza Valero S.L.N.E,Retail,China,Enterprise,2078,2024-08-20,2025-02-16,True,2025-01-13,87,40
2997,924354ec-155c-4c60-a10f-010dbcec6bbb,Reig y asociados S.L.,Finance,Mali,Starter,249,2025-12-05,2026-11-30,True,2026-01-28,1519,17
2998,c66d1228-e4f1-40f3-a248-9c67f3974add,Beltrán & Asociados S.A.,Health,Italia,Enterprise,3038,2023-03-03,2023-06-01,True,2023-03-24,3439,8


In [257]:
l_cust = int(len(df_customers))
indices = np.random.choice(l_cust, size=int(l_cust*0.05), replace=False)
df_customers.loc[indices, "country"] = None

In [258]:
indices = np.random.choice(l_cust, size=int(l_cust*0.08), replace=False)
df_customers.loc[indices, "employee_count"] = np.nan

In [259]:
indices = np.random.choice(l_cust, size=int(l_cust*0.03), replace=False)
df_customers.loc[indices, "industry"] = None

In [260]:
df_customers.isna().sum()

customer_id          0
company_name         0
industry            90
country            150
plan_type            0
mrr                  0
contract_start       0
contract_end         0
churned              0
churned_date      1495
employee_count     240
health_score         0
dtype: int64

In [103]:
df_customers.to_csv('../data/raw/customers.csv', index=False)

## Generate Subscriptions
- sub_id
- customer_id
- plan
- billing_cycle
- start_date
- end_date
- discount_pct
- payment_method
- last_payment_status

In [ ]:
def generar_suscripcion(cliente: dict):
    if 12 > ((cliente["contract_end"] - cliente["contract_start"]).days / 30):
        billing_cycle = "Mensual"
    else: 
        billing_cycle = "Anual"

    r = random.randint(0,100)
    if cliente["churned"] == True:
        
        if r < 40:
            last_payment_status = "failed"
        elif 40 <= r and r <= 80:
            last_payment_status = "pending"
        else:
            last_payment_status = "success"
    else:
        if r < 70:
            last_payment_status = "success"
        elif 70 <= r and r <= 85:
            last_payment_status = "pending"
        else:
            last_payment_status = "failed"
    
    r = random.randint(0,100)
    if r < 5:
        discount_pct = 30
    elif 5 <= r and r <= 15:
        discount_pct = random.choice([20,30])
    else:
        discount_pct = 0

    return {
        "sub_id":               fake.uuid4(),
        "customer_id":          cliente["customer_id"],
        "plan":                 cliente["plan_type"],
        "billing_cycle":        billing_cycle,
        "start_date":           cliente["contract_start"],
        "end_date":             cliente["contract_end"],
        "discount_pct":         discount_pct,       
        "payment_method":       random.choice(["tarjeta de credito","transferencia bancaria","PayPal"]),      
        "last_payment_status":  last_payment_status
    }

In [149]:
subscriptions = [generar_suscripcion(cust) for cust in df_customers.to_dict('records')]

In [151]:
df_subscriptions = pd.DataFrame(subscriptions)
df_subscriptions

,sub_id,customer_id,plan,billing_cycle,start_date,end_date,discount_pct,payment_method,last_payment_status
0,5bfdb261-0de4-4381-8593-6b1e7dd2942e,949c8a0e-c8f2-40cc-8f97-527bcd35406a,Enterprise,Mensual,2023-12-19,2024-03-18,success,PayPal,pending
1,20ef23d8-ad86-42a5-87b6-4b489137925f,bb69a3c1-816c-454b-8d94-481186e8c99a,Starter,Anual,2025-07-26,2026-07-21,success,PayPal,success
2,79ddbdea-95c7-4a68-b464-07769d3f6ff3,5220a66c-36b8-4037-8c43-b08eb81203e2,Enterprise,Mensual,2024-02-23,2024-03-24,success,PayPal,success
3,1b3dddce-e76e-4648-85bc-29a6b23de5c3,b6878b37-e5da-460d-9d8b-aafa4eb5b029,Pro,Anual,2022-08-13,2023-08-08,success,PayPal,failed
4,9ce0b582-6682-4631-9b79-eada81a6c3a4,3439ad77-7bbb-41c9-88f8-1f276cf29283,Pro,Mensual,2023-01-31,2023-07-30,success,transferencia bancaria,success
...,...,...,...,...,...,...,...,...,...
2995,41d51b53-51cb-40bf-976e-3cb2b7721ad3,24d727ee-2ce0-4db9-a64f-e82458f8a86f,Enterprise,Mensual,2023-01-17,2023-07-16,success,tarjeta de credito,pending
2996,fb5e1001-e39f-4bb8-9e5a-748fee0c04a4,dae3f348-c69f-4eae-bbe0-c5e393987c9b,Enterprise,Mensual,2024-08-20,2025-02-16,success,PayPal,failed
2997,b0b7b843-f993-4440-9424-8a8d2f407641,924354ec-155c-4c60-a10f-010dbcec6bbb,Starter,Anual,2025-12-05,2026-11-30,success,tarjeta de credito,failed
2998,58ba2576-8cb0-4062-8b99-2bfac9b93095,c66d1228-e4f1-40f3-a248-9c67f3974add,Enterprise,Mensual,2023-03-03,2023-06-01,success,transferencia bancaria,pending


In [152]:
df_subscriptions.to_csv('../data/raw/subscriptions.csv', index=False)

## Usage Logs
- log_id
- customer_id
- user_id
- event_date
- feature_used
- session_duration_min
- login_count
- actions_count
- device_type

In [224]:
def generar_logs_cliente(cliente: dict):
    logs = []
    if cliente["churned"] == True: 
        n_logs = random.randint(10,80)
    else:
        n_logs = random.randint(50,100)

    if cliente["plan_type"] == "Starter":
        users = [fake.uuid4() for _ in range(np.random.randint(1,6))]
    elif cliente["plan_type"] == "Pro":
        users = [fake.uuid4() for _ in range(np.random.randint(5,21))]
    elif cliente["plan_type"] == "Enterprise":
        users = [fake.uuid4() for _ in range(np.random.randint(20,51))]

    modules = ["dashboard","task_manager","reports","integrations","billing","team_settings","calendar","notifications"]

    

    for _ in range(n_logs):
        if cliente["churned"] == True: 
            event_date = fake.date_between(start_date=cliente["contract_start"],end_date=cliente["churned_date"])
            limit_day = (cliente['churned_date'] - event_date).days
            if  limit_day <= 28 and limit_day > 21:
                session_duration_min = random.randint(1,31)
                actions_count = random.randint(1,21)
            elif  limit_day <= 20 and limit_day > 14:
                session_duration_min = random.randint(1,23)
                actions_count = random.randint(1,16)
            elif  limit_day <= 14 and limit_day > 7:
                session_duration_min = random.randint(1,16)
                actions_count = random.randint(1,11)
            elif  limit_day <= 7 and limit_day > 0:
                session_duration_min = random.randint(1,9)
                actions_count = random.randint(1,5)
            else:
                session_duration_min = random.randint(1,41)
                actions_count = random.randint(1,31)
        else:
            event_date = fake.date_between(start_date=cliente["contract_start"],end_date=cliente["contract_end"])
            session_duration_min = random.randint(10,120)
            actions_count = random.randint(10,101)
            

        logs.append({
            "log_id":               fake.uuid4(),
            "customer_id":          cliente["customer_id"],
            "user_id":              random.choice(users),
            "event_date":           event_date,
            "feature_used":         random.choice(modules),
            "session_duration_min": session_duration_min,
            "login_count":          random.randint(1,5),
            "actions_count":        actions_count,
            "device_type":          random.choice(["desktop","mobile","tablet"])
        })

    return logs

In [225]:
logs = [generar_logs_cliente(customer) for customer in df_customers.to_dict('records')]

In [226]:
logs_flatten = [log for cliente_logs in logs for log in cliente_logs]

In [227]:
df_logs = pd.DataFrame(logs_flatten)
df_logs

,log_id,customer_id,user_id,event_date,feature_used,session_duration_min,login_count,actions_count,device_type
0,b145d661-f327-4502-854d-64702b7281b2,949c8a0e-c8f2-40cc-8f97-527bcd35406a,10001b85-7e5f-4ecd-a085-c341f6d44936,2024-01-07,notifications,11,4,6,mobile
1,99365c96-429c-4c13-b7a3-85d64dcfebcd,949c8a0e-c8f2-40cc-8f97-527bcd35406a,c8f7eea7-c05d-40d1-a309-d28535883da7,2024-01-13,integrations,8,3,1,mobile
2,7ba86ac1-502d-40c3-ae84-0742b583ae2c,949c8a0e-c8f2-40cc-8f97-527bcd35406a,70ff4a43-33b2-4e70-989a-4ed2552c1a5a,2024-01-07,integrations,9,4,2,tablet
3,dbc44d59-8a5a-42c8-8c03-fadef1b95948,949c8a0e-c8f2-40cc-8f97-527bcd35406a,b2668aed-47c8-4f21-bdc3-c3010d1ff92f,2024-01-01,dashboard,2,5,8,mobile
4,428f8dd4-5e0a-4399-abee-5d685bedf72e,949c8a0e-c8f2-40cc-8f97-527bcd35406a,58ccfde5-bd19-4a52-b698-ffa633a32c16,2024-01-12,billing,3,3,4,tablet
...,...,...,...,...,...,...,...,...,...
179877,d53308f9-6428-45a9-b2c1-5a59c3459f22,80cc940c-d4d0-427d-9efb-98a97e68259f,1925b855-8981-4f14-902e-9f10fe69c0d5,2022-07-30,calendar,27,3,18,desktop
179878,a06f5fa0-4611-4875-b843-4d00452766ab,80cc940c-d4d0-427d-9efb-98a97e68259f,47826f9c-7e31-4bfa-89e5-8b5481e12650,2022-08-09,calendar,15,5,27,desktop
179879,0976cc13-e052-4b3d-ae1d-f86736245dc8,80cc940c-d4d0-427d-9efb-98a97e68259f,17d14198-bdb4-40a7-ad39-2c17d6de3f8e,2022-09-16,dashboard,15,5,2,mobile
179880,b786026c-73c1-4d47-9b14-8e5582b517d9,80cc940c-d4d0-427d-9efb-98a97e68259f,47826f9c-7e31-4bfa-89e5-8b5481e12650,2022-09-11,dashboard,17,3,7,desktop


In [261]:
l_logs = int(len(df_logs))
indices = np.random.choice(l_logs, size=int(l_logs*0.03), replace=False)
df_logs.loc[indices, "session_duration_min"] = np.nan

In [262]:
df_logs.isna().sum()

log_id                     0
customer_id                0
user_id                    0
event_date                 0
feature_used               0
session_duration_min    5396
login_count                0
actions_count              0
device_type                0
dtype: int64

In [228]:
df_logs.to_csv('../data/raw/usage_logs.csv', index=False)

## Support Tickets
- ticket_id
- customer_id
- created_at
- resolved_at
- category
- priority
- sentiment_score
- csat_score
- escalated

In [206]:
def generar_tickets(cliente: dict):
    if cliente["churned"] == True:
        tickets_count = random.randint(5,21)
    else:
        tickets_count = random.randint(1,9)

    tickets = []
    for _ in range(tickets_count):
        create_at = fake.date_between(start_date=cliente["contract_start"], end_date=cliente["contract_end"])
        if cliente["churned"] == True:
            r = random.randint(0,100)
            if r > 70:
                resolved_at = fake.date_between(start_date=create_at, end_date=cliente["contract_end"])
            elif r <= 70:
                resolved_at = None
            
            sentiment_score = random.uniform(-1,0.2)

            csat_score = random.randint(1,3)

            r = random.randint(0,100)

            if r > 60:
                escalated  = True
            elif r <= 60:
                escalated  = False

        else:
            r = random.randint(0,100)
            if r > 85:
                resolved_at = None
            elif r <= 85:
                resolved_at = fake.date_between(start_date=create_at, end_date=cliente["contract_end"])
            
            sentiment_score = random.uniform(-0.2, 1)

            csat_score = random.randint(2,5)

            r = random.randint(0,100)

            if r > 90:
                escalated  = True
            elif r <= 90:
                escalated  = False

        tickets.append(
        {
            "ticket_id":        fake.uuid4(),
            "customer_id":      cliente["customer_id"],
            "create_at":        create_at,
            "resolved_at":      resolved_at,
            "category":         random.choice(["billing", "technical", "onboarding", "feature_request", "bug"]),
            "priority":         random.choice(["low", "medium", "high", "critical"]),
            "sentiment_score":  sentiment_score,
            "csat_score":       csat_score,
            "escalated":        escalated
        }
        )
    
    return tickets


In [207]:
tickets = [generar_tickets(customer) for customer in df_customers.to_dict('records')]

In [209]:
tickets_flatten = [ticket for tickets_cust in tickets for ticket in tickets_cust]

In [211]:
df_tickets = pd.DataFrame(tickets_flatten)
df_tickets

,ticket_id,customer_id,create_at,resolved_at,category,priority,sentiment_score,csat_score,escalated
0,53f17ab4-0e04-40b8-8210-ec1a12c56925,949c8a0e-c8f2-40cc-8f97-527bcd35406a,2024-03-06,None,feature_request,low,-0.746452,2,False
1,c8c16b8e-30f8-4b79-a484-9ff1358e664b,949c8a0e-c8f2-40cc-8f97-527bcd35406a,2024-01-01,2024-02-29,onboarding,critical,0.085689,1,False
2,e1ec6198-3741-4219-85ee-1a02514934d1,949c8a0e-c8f2-40cc-8f97-527bcd35406a,2023-12-22,None,onboarding,high,-0.861923,3,False
3,f6c3cb6d-29a3-4a5c-8500-25d143398fe1,949c8a0e-c8f2-40cc-8f97-527bcd35406a,2024-02-11,None,billing,medium,-0.283265,2,False
4,9ee866bd-562f-4d3c-b0fe-a1862a7b8d4c,949c8a0e-c8f2-40cc-8f97-527bcd35406a,2024-02-09,2024-02-12,billing,critical,-0.441554,2,True
...,...,...,...,...,...,...,...,...,...
27139,c6e4992d-7e14-4f2a-8538-bdab60e0292f,80cc940c-d4d0-427d-9efb-98a97e68259f,2022-08-06,None,billing,critical,-0.418279,3,False
27140,49f279d8-3123-4b11-99dc-70ab4e67f1e6,80cc940c-d4d0-427d-9efb-98a97e68259f,2022-09-17,None,billing,critical,0.107800,3,False
27141,80d22464-3fbc-4352-be34-bc422748810f,80cc940c-d4d0-427d-9efb-98a97e68259f,2022-09-11,None,onboarding,high,-0.895275,3,False
27142,18208a07-37d4-4ce9-aad8-5b4386b6d75e,80cc940c-d4d0-427d-9efb-98a97e68259f,2022-09-18,2022-09-29,bug,high,-0.148289,1,True


In [263]:
l_tickets = int(len(df_tickets))
indices = np.random.choice(l_tickets, size=int(l_tickets*0.15), replace=False)
df_tickets.loc[indices, "csat_score"] = np.nan

In [264]:
indices = np.random.choice(l_tickets, size=int(l_tickets*0.10), replace=False)
df_tickets.loc[indices, "sentiment_score"] = np.nan

In [265]:
df_tickets.isna().sum()

ticket_id              0
customer_id            0
create_at              0
resolved_at        14804
category               0
priority               0
sentiment_score     2714
csat_score          4071
escalated              0
dtype: int64

In [212]:
df_tickets.to_csv('../data/raw/support_tickets.csv', index = False)